In [ ]:
import os
import platform
import shutil
import subprocess
import tempfile
from pathlib import Path

print('OS:', platform.platform())
print('Architecture:', platform.machine())
print('Logical CPUs:', os.cpu_count())
cpuinfo = Path('/proc/cpuinfo')
cpu_model = next((line.split(':', 1)[1].strip() for line in cpuinfo.read_text(errors='replace').splitlines() if line.lower().startswith('model name')), 'not reported') if cpuinfo.exists() else platform.processor() or 'not reported'
print('CPU model:', cpu_model)

meminfo = Path('/proc/meminfo')
if meminfo.exists():
    values = {line.split(':', 1)[0]: line.split(':', 1)[1].strip() for line in meminfo.read_text(errors='replace').splitlines() if ':' in line}
    print('RAM total:', values.get('MemTotal', 'not reported'))
    print('RAM available:', values.get('MemAvailable', 'not reported'))
else:
    print('RAM: not reported by /proc/meminfo')

working = Path('/kaggle/working')
print('Working directory exists:', working.is_dir())
if working.is_dir():
    disk = shutil.disk_usage(working)
    print('Working disk free bytes:', disk.free)
else:
    print('Working disk free bytes: not available')

nvidia_smi = shutil.which('nvidia-smi')
if nvidia_smi:
    result = subprocess.run([nvidia_smi, '-L'], capture_output=True, text=True, timeout=5, check=False)
    print('GPU query exit code:', result.returncode)
    print('GPU query output:', (result.stdout or result.stderr).strip() or 'no device reported')
else:
    print('GPU status: nvidia-smi not available; allocation not confirmed by this check')
print('TPU device paths:', [str(path) for path in Path('/dev').glob('accel*')])
print('Kaggle notebook UI accelerator selection must also be checked manually; this cell does not request an accelerator.')

candidates = [Path(path) for path in (shutil.which('blender'), '/usr/bin/blender', '/usr/local/bin/blender', '/opt/blender/blender', str(Path.home() / 'blender' / 'blender')) if path]
blender = next((path for path in candidates if path.is_file() and os.access(path, os.X_OK)), None)
print('Blender executable:', str(blender) if blender else 'not found in PATH or checked standard locations')
if blender:
    result = subprocess.run([str(blender), '--background', '--version'], capture_output=True, text=True, timeout=20, check=False)
    print('Blender background/version exit code:', result.returncode)
    print('Blender background/version output:', (result.stdout or result.stderr).strip())

if working.is_dir():
    expected = 'AI-Engine harmless temporary write/read check'
    with tempfile.TemporaryDirectory(prefix='ai-engine-preflight-', dir=working) as temporary_directory:
        test_file = Path(temporary_directory) / 'write-read-check.txt'
        test_file.write_text(expected, encoding='utf-8')
        actual = test_file.read_text(encoding='utf-8')
        print('Temporary write/read path:', str(test_file))
        print('Temporary write/read verified:', actual == expected)
else:
    print('Temporary write/read not attempted: /kaggle/working is unavailable')
